# Stage1 × Cboe VIX/VIX3M 期限结构来源门 v2

仅修复上一实际 Qlib 会话的日历左边界；原始 Cboe 字节、公式、来源用途及无收益/拟合/账户边界不变。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_cboe_vix_term_source_v2_20261023';plan=json.loads((card/'plan.json').read_text());snap=root/'data/snapshots/stage1_cboe_vix_term_source_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['budget']['model_fits']==plan['budget']['native_accounts']==plan['budget']['SOXX_outcome_reads']==0
nas_source=json.loads((root/plan['source_archive_receipt']).read_text());assert nas_source['remote_scoped_restore_verified']
series={}
for key in ('vix','vix3m'):
 spec=plan['source_files'][key];path=root/spec['relative_path'];assert sha(path)==spec['sha256'] and path.stat().st_size==spec['size_bytes']
 f=pd.read_csv(path);assert list(f.columns)==['DATE','OPEN','HIGH','LOW','CLOSE'];f.DATE=pd.to_datetime(f.DATE,format='%m/%d/%Y',errors='raise')
 assert not f.DATE.isna().any() and f.DATE.is_unique and f.DATE.is_monotonic_increasing and np.isfinite(f.CLOSE).all() and (f.CLOSE>0).all()
 series[key]=f.set_index('DATE').CLOSE.astype(float).rename(key)
source=pd.concat(series,axis=1,join='inner');source['term_log_vix_over_vix3m']=np.log(source.vix/source.vix3m)
assert source.index.is_unique and source.index.is_monotonic_increasing and np.isfinite(source.term_log_vix_over_vix3m).all()
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
all_cal=pd.DatetimeIndex(D.calendar(start_time='2010-12-01',end_time=plan['cohorts']['full_signal'][1],freq='day'))
cal=all_cal[all_cal>=pd.Timestamp(plan['cohorts']['full_signal'][0])]
assert len(cal)==3966 and cal.is_unique and cal.is_monotonic_increasing and all_cal[all_cal.get_loc(cal[0])-1]==pd.Timestamp('2010-12-31')
previous=pd.Series(all_cal,index=all_cal).shift(1).reindex(cal);mapped=pd.DataFrame(index=cal);mapped.index.name='datetime';mapped['source_date']=previous
mapped['Cboe_VIX_close']=previous.map(source.vix);mapped['Cboe_VIX3M_close']=previous.map(source.vix3m)
mapped['Cboe_VIX_term_log_ratio']=np.log(mapped.Cboe_VIX_close/mapped.Cboe_VIX3M_close)
mapped['source_known']=mapped.Cboe_VIX_term_log_ratio.notna()
assert (mapped.loc[mapped.source_known,'source_date']<mapped.loc[mapped.source_known].index).all()
assert np.isfinite(mapped.loc[mapped.source_known,'Cboe_VIX_term_log_ratio']).all()
parent=root/'research/minute_information_pit_allocator_20261012';parent_summary=json.loads((parent/'summary.json').read_text())
with R.uri_context(str(parent/'qlib_recorder')):
 old=R.get_recorder(recorder_id='68e31a0a0fad44e7ba1d1f7d856c7a0d',experiment_id='905520356740140469')
 original=old.load_object('models/base_outer_71');base=old.load_object('inputs/features')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns) and len(base)==len(cal) and base.index.equals(cal)
train=(mapped.index>=pd.Timestamp(plan['cohorts']['train'][0]))&(mapped.index<=pd.Timestamp(plan['cohorts']['train'][1]));held=(mapped.index>=pd.Timestamp(plan['cohorts']['held_reused'][0]))&(mapped.index<=pd.Timestamp(plan['cohorts']['held_reused'][1]))
correlations={};valid=train&mapped.source_known
for name in base_names:
 x=base[name].astype(float).loc[valid].to_numpy();y=mapped.loc[valid,'Cboe_VIX_term_log_ratio'].to_numpy();mask=np.isfinite(x)&np.isfinite(y)
 if mask.sum()>100 and np.std(x[mask])>0 and np.std(y[mask])>0:correlations[name]=float(np.corrcoef(x[mask],y[mask])[0,1])
assert len(correlations)>30
static=pd.concat({'feature':mapped[['Cboe_VIX_term_log_ratio']],'meta':mapped[['source_date','source_known']]},axis=1)
static.index=pd.MultiIndex.from_arrays([mapped.index,['SOXL_SOXX_CASH']*len(mapped)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train']),'held':tuple(plan['cohorts']['held_reused'])})
tr=dataset.prepare('train',col_set=['feature','meta']);he=dataset.prepare('held',col_set=['feature','meta'])
assert len(tr)==int(train.sum())==3020 and len(he)==int(held.sum())==946
missing=mapped.index[~mapped.source_known]
audit={'source_rows':{k:len(v) for k,v in series.items()},'first_source_date':{k:str(v.index.min().date()) for k,v in series.items()},'last_source_date':{k:str(v.index.max().date()) for k,v in series.items()},'common_rows':len(source),'full_calendar_rows':len(mapped),'train_calendar_rows':int(train.sum()),'held_calendar_rows':int(held.sum()),'train_known_rows':int(mapped.loc[train,'source_known'].sum()),'held_known_rows':int(mapped.loc[held,'source_known'].sum()),'missing_signal_dates':[str(x.date()) for x in missing],'train_original39_pairwise_max_abs_Pearson':max(abs(v) for v in correlations.values()),'train_original39_valid_correlations':len(correlations),'source_day_later_than_signal':False,'historical_first_public_or_revision_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
tracking=card/'qlib_recorder'
assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id)
 dataset.config(dump_all=True,recursive=True)
 rec.save_objects(artifact_path='inputs',source_series=source,source_files=plan['source_files'],dataset=dataset)
 rec.save_objects(artifact_path='analysis',mapped_signals=mapped,source_audit=audit,training_correlations=correlations)
 rec.log_metrics(train_source_known=audit['train_known_rows'],held_source_known=audit['held_known_rows'],source_only=1)
summary={'experiment':plan['experiment'],'decision':'conditional_Cboe_VIX_term_source_mapped_first_public_unqualified' if not missing.size else 'source_coverage_gap_no_information_fit','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'source_nas_archive_sha256':nas_source['sha256'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'audit':audit,'artifact_policy':plan['artifact_policy'],'new_model_fits':0,'new_native_accounts':0,'independent_forward_qualified_days':0,'next_action':'Independent source review, then separately freeze one original39 matched information screen only if full calendar and training novelty/source status permit; no account before historical first-public/next-decision availability qualification.'}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='cboe-vix-term-restore-') as t:
 archive=Path(t)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as z:
  names=[m.name for m in z.getmembers() if m.isfile()]
  assert any(n.endswith('/artifacts/analysis/mapped_signals') for n in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':evidence['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print({k:audit[k] for k in ('full_calendar_rows','train_known_rows','held_known_rows','train_original39_pairwise_max_abs_Pearson')})


2026/10/23 06:38:52 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[7645:MainThread](2026-10-23 06:38:53,083) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[7645:MainThread](2026-10-23 06:38:53,085) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[7645:MainThread](2026-10-23 06:38:53,085) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


...

[7645:MainThread](2026-10-23 06:38:53,698) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_cboe_vix_term_source_v2_20261023.


[7645:MainThread](2026-10-23 06:38:53,700) INFO - qlib.workflow - [exp.py:258] - Experiment 742837079633922141 starts running ...


[7645:MainThread](2026-10-23 06:38:53,769) INFO - qlib.workflow - [recorder.py:345] - Recorder 306545c51d9549a4ab68e37a33a04dde starts running under Experiment 742837079633922141 ...


[7645:MainThread](2026-10-23 06:38:54,385) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'full_calendar_rows': 3966, 'train_known_rows': 3020, 'held_known_rows': 946, 'train_original39_pairwise_max_abs_Pearson': 0.6317475475077795}


## 中文结论

# Cboe VIX/VIX3M 期限结构：来源完整，候选拟合前关闭

两份 Cboe 官方日线原始 CSV 分别有 9,299 与 4,300 行。限定修复后的原生 Qlib 日历把上一实际交易日的 `log(VIX close / VIX3M close)` 映射到原始 Stage1 的全部 3,966 个信号日，训练 3,020/3,020、复用评价 946/946 有数值；训练期对原 39 项输入最大逐项相关绝对值 0.6317。v1 首日漏值只是日历左边界错误，原始 2010-12-31 两文件均有值；v1 失败和 Recorder 保留。

**有数值不等于历史可用。** Cboe 方法文件给 VIX3M 的首次值为 2009 年 9 月、推出时间为 2013 年 10 月。原训练段有 691 日使用确定早于 2013 年 10 月的回算值，另有 23 个推出月份信号日缺精确首发日；即使其余日有当前 CSV 数值，也未证明原版在下一次 09:46 决策前公开或未被修订。两份文件在本次收取时更新不同步，不能反推历史同日可用。

独立去重审查发现，已关闭的 `cboe_h5_information_20261009` 用上一会话的 `log(VIX9D/VIX3M)` 做过匹配 H5 期限结构信息检验并在开发/复用历史上失败。本候选将短端改为约 30 日 VIX、仍以三个月为分母且保留 H5 接收任务，主要是失败后的期限选择。因此决定 `source_map_verified_candidate_closed_before_fit_duplicate_term_horizon_and_prelaunch_backfill`：**0 收益读取、0 拟合、0 新资金账户**，不以原 Stage1 39 项接收器重包装同一经济机制。

原始 CSV 与响应元数据在 Yanxuan NAS 的 SHA-256 为 `fd9562a34727d30617d4c103f513056769c4ec9e5deed1f9cfee8fc72f27e5ec`；v2 原生 Recorder `306545c51d9549a4ab68e37a33a04dde` 的 NAS SHA-256 为 `bd8842b2f34363bc42cf7e1ee8876da18c8b21b16b194f1ebaf9284ae817cdff`。两组都经远端哈希及限定恢复，Recorder 已定向读回。整体 Stage1 0bp/SOXX 目标未达成；下一步必须寻找实质不同的决策时机制。

[Cboe VIX 历史文件](https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv) · [Cboe VIX3M 历史文件](https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX3M_History.csv) · [Cboe 首次值与推出年月](https://cdn.cboe.com/api/global/us_indices/governance/Volatility_Index_Methodology_Selected_SPX_Target_Expected_Volatility_Term_Indices.pdf)
